In [ ]:
pip install watchdog nbclient


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import time
import os
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
from nbclient import NotebookClient
from nbformat import read
import time
import json
from nbconvert import PythonExporter
import types

In [ ]:
# 📦 Importer dynamiquement staging_loader_test.ipynb
def import_functions_from_notebook(path):
    with open(path, encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)
    exporter = PythonExporter()
    source_code, _ = exporter.from_notebook_node(nb)
    module = types.ModuleType("staging_module")
    exec(source_code, module.__dict__)
    return module

staging_module = import_functions_from_notebook("staging_loader_test.ipynb")

# 🔁 Récupérer les fonctions utiles
load_and_historize_with_log = staging_module.load_and_historize_with_log
conn = staging_module.conn
referentiels = staging_module.referentiels

In [ ]:
# 📁 Dossier à surveiller
watched_folder = "../Data"

# 📓 Notebook à exécuter automatiquement
notebook_path = "staging_loader_test.ipynb"


last_run_time = 0
cooldown_seconds = 5  # Pour éviter les doublons rapides

class ChangeHandler(FileSystemEventHandler):
    def on_modified(self, event):
        global last_run_time
        if event.src_path.endswith(".csv") and not os.path.basename(event.src_path).startswith(".~"): #Fichier temporaire sera ignoré
            now = time.time()
            if now - last_run_time > cooldown_seconds:
                last_run_time = now
                csv_name = os.path.basename(event.src_path)
                run_targeted_table(csv_name)



def run_targeted_table(csv_name):
    print(f"\n🚀 Déclenchement ciblé pour : {csv_name}")
    try:
        with open("staging_config.json", encoding="utf-8") as f:
            configs = json.load(f)
        configs = [cfg for cfg in configs if cfg.get("csv_name") == csv_name]

        if not configs:
            print(f"⚠️ Aucune configuration trouvée pour {csv_name}")
            return

        for config in configs:
            print(f"📥 Traitement de la table : {config['table']}")
            load_and_historize_with_log(conn, config, referentiels)

    except Exception as e:
        print(f"❌ Erreur pendant le chargement de {csv_name} : {e}")






18:12:52 | 👀 Surveillance active du dossier : ../Data
⏳ Laisse ce notebook tourner pour activer l'automatisation...
18:13:12 | 📥 Nouveau fichier détecté : ../Data\.~joueurs.csv
18:13:12 | ⏳ Attente pour sécuriser la lecture du fichier modifié...
18:13:14 | 🚀 Lancement de staging_loader_test.ipynb ...
18:14:36 | ✅ Pipeline exécuté avec succès.
18:14:36 | 🔁 Fichier modifié : ../Data\.~joueurs.csv
18:14:36 | ⏳ Attente pour sécuriser la lecture du fichier modifié...
18:14:38 | 🚀 Lancement de staging_loader_test.ipynb ...
18:16:01 | ✅ Pipeline exécuté avec succès.
18:16:01 | 🔁 Fichier modifié : ../Data\joueurs.csv
18:16:01 | ⏳ Attente pour sécuriser la lecture du fichier modifié...
18:16:03 | 🚀 Lancement de staging_loader_test.ipynb ...
18:17:26 | ✅ Pipeline exécuté avec succès.
18:17:26 | 📥 Nouveau fichier détecté : ../Data\.~joueurs.csv
18:17:26 | ⏳ Attente pour sécuriser la lecture du fichier modifié...
18:17:28 | 🚀 Lancement de staging_loader_test.ipynb ...
18:18:52 | ✅ Pipeline exécuté

In [ ]:
# 🚨 Démarrer l'observation
observer = Observer()
observer.schedule(ChangeHandler(), path=watched_folder, recursive=False)
observer.start()

print(f"👁️ Surveillance activée sur le dossier : {watched_folder}")
print("⏳ En attente de modifications ...")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    observer.stop()
observer.join()